In [1]:
%reload_ext autoreload
%autoreload 2

import numpy as np
import os, sys
from astropy.io import fits
import healpy as hp
from decimal import Decimal

from scipy.interpolate import interp1d
from scipy.optimize import minimize, shgo
from scipy.stats import poisson
from scipy import signal

sys.path.append('..')

import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc_file('../notebooks/matplotlibrc')

In [2]:
from utils.cart import to_cart

## 0. Config

In [3]:
ebins = 2*np.logspace(-1, 3, 41)
# de = ebins[1:] - ebins[:-1]
# emid = 10**((np.log10(ebins[1:]) + np.log10(ebins[:-1]))/2.)
print([f'{ebin:.1f}' for ebin in ebins[10:20]])

['2.0', '2.5', '3.2', '4.0', '5.0', '6.3', '8.0', '10.0', '12.6', '15.9']


## 1. Diffuse templates

In [4]:
TMPL_DIR = '/zfs/tslatyer/fermimaps/allsky/PASS8_Jun19_UltracleanVeto/specbin'

In [5]:
for diff_name in ['p6v11', 'ccwaics', 'ccwapibrem', 'ccwfics', 'ccwfpibrem']:
    
    print(diff_name, end=' ')

    nsideload = 512
    npixload = hp.nside2npix(nsideload)
    pixarea = hp.nside2pixarea(nsideload)

    # Setup energy bins
#     ebins_str = ['0002.0','0002.5', '0003.2','0004.0','0005.0',
#                  '0006.3','0008.0','0010.0','0012.6','0015.9','0020.0']
    ebins = 2*np.logspace(-1, 3, 41)
    de = ebins[1:] - ebins[:-1]
    emid = 10**((np.log10(ebins[1:]) + np.log10(ebins[:-1]))/2.)

    # Get the correct bins
    de = de[10:20]
    emid = emid[10:20]

    # init
    diff = np.zeros(npixload)

    diffload = fits.open(f'{TMPL_DIR}/diffuse_model_map_healpix_{diff_name}_PASS8_Jun19_UltracleanVeto_bestpsf.fits')
    diffall = diffload[0].data

    for i in range(10):
        # Diff has all 40, 2 GeV is i=10, so start there
        # NB!!! The diffuse model has different units to everything else.
        # It is already exposure corrected, but is divided by the bin size in MeV
        # I'm here copying the approach in the fermi plugin
        diff += diffall[:, i+10] * de[i] * 4.*np.pi/npixload * 1e3

    # Change nside to output value
    nside = nsideload
    npix = hp.nside2npix(nside)
    diff = hp.ud_grade(diff, nside, power=-2)

    # Adjust templates to all have mean 1 in the ROI
    # ROI: |b| >= 2, |r| <= 30
    theta, phi = hp.pix2ang(nside, range(npix))
    larr = np.zeros(npix)
    for i in range(npix):
        #larr[i] = pm.mod(phi[i]+np.pi,2*np.pi)-np.pi
        larr[i] = ((phi[i]+np.pi) % (2*np.pi)) - np.pi
    barr = np.pi/2 - theta
    rarr = np.arccos(np.cos(larr)*np.cos(barr))

    roi = np.where((np.abs(barr) >= 2*np.pi/180.) & (rarr <= 30*np.pi/180.))[0]

    diff /= np.mean(diff[roi])

    np.save(f'data/templates_Jun19/{diff_name}_NSIDE512.npy', diff)

p6v11 ccwaics ccwapibrem ccwfics ccwfpibrem 

## 2. Other templates

In [6]:
# get filenames and eng labels
fns = [fn for fn in os.listdir(TMPL_DIR) if 'fwhm000-0512-bestpsf-nopsc' in fn]
eng_str_s = [fn.split('GeV')[0].split('-')[-1] for fn in fns]
eng_eng_str_s = [(float(eng_str), eng_str) for eng_str in eng_str_s]
eng_eng_str_s.sort()

eng_lstr_s = [fn.split('GeV')[0].split('-')[-2] for fn in fns]
eng_eng_lstr_s = [(float(eng_str), eng_str) for eng_str in eng_lstr_s]
eng_eng_lstr_s.sort()

eng_eng_str_s.insert(0, eng_eng_lstr_s[0])

In [7]:
# First load data and exposure
# 0 entry in units of [counts/cm^2/s/sr]
# exposure in [cm^2 s]

nsideload = 512
npixload = hp.nside2npix(nsideload)
pixarea = hp.nside2pixarea(nsideload)

# Setup energy bins
ebins_str = [eng_str for eng, eng_str in eng_eng_str_s[10:21]]
ebins = 2*np.logspace(-1,3,41)
de = ebins[1:] - ebins[:-1]
emid = 10**((np.log10(ebins[1:]) + np.log10(ebins[:-1]))/2.)

# Get the correct bins
de = de[10:20]
emid = emid[10:20]

# Load maps
counts = np.zeros(npixload)
expall = np.zeros(shape=(npixload,10))
pscmdl = np.zeros(npixload)

for i in range(10):
    load = fits.open(f'{TMPL_DIR}/fermi-allsky-{ebins_str[i]}-{ebins_str[i+1]}GeV-fwhm000-0512-bestpsf-nopsc.fits')
    fi = load[0].data
    ei = load[1].data
    countsi = fi*ei*pixarea # [counts/pixel]
    counts += countsi
    expall[:,i] = ei

    load = fits.open(f'{TMPL_DIR}/fermi-allsky-{ebins_str[i]}-{ebins_str[i+1]}GeV-fwhm000-0512-bestpsf-pscmdl.fits')
    pi = load[0].data

    pscmdl += pi*ei

# Round counts to be exact, some values slightly off an integer in above
counts = np.round(counts)
exp = np.mean(expall, axis=1)

# Load ps mask
load = fits.open('/zfs/nrodd/MakeNPTFitData/PtSourceMask/Allpscmask_3FGL-energy2.00000_0.95_ULTRACLEANVETO_bestpsf.fits') 
psmask = load[0].data

# Change nside to output value
nside = 512
npix = hp.nside2npix(nside)
#counts = hp.ud_grade(counts, nside, power=-2)
#exp = hp.ud_grade(exp, nside, power=0)
#diff = hp.ud_grade(diff, nside, power=-2)
#pscmdl = hp.ud_grade(pscmdl, nside, power=-2)

# Output exposure before any further manipulations, to avoid its value getting reset
np.save('data/fermidata/exposure_Jun19_NSIDE512.npy', exp)

# Load the other templates - these are nside 256
iso = exp
temps = fits.open('/zfs/nrodd/Templates/template_finenfw3.fits')[1].data
bub = hp.ud_grade(temps[9][4], nside, power=-2) * exp
nfw = hp.ud_grade(temps[195][4], nside, power=-2) * exp
disk = hp.ud_grade(fits.open('/zfs/nrodd/MakeNPTFitData/raw_disk.fits')[0].data, nside, power=-2) * exp

# Adjust templates to all have mean 1 in the ROI
# ROI: |b| >= 2, |r| <= 30
theta, phi = hp.pix2ang(nside, range(npix))
larr = np.zeros(npix)
for i in range(npix):
    #larr[i] = pm.mod(phi[i]+np.pi,2*np.pi)-np.pi
    larr[i] = ((phi[i]+np.pi) % (2*np.pi)) - np.pi
barr = np.pi/2 - theta
rarr = np.arccos(np.cos(larr)*np.cos(barr))

roi = np.where((np.abs(barr) >= 2*np.pi/180.) & (rarr <= 30*np.pi/180.))[0]

#diff /= np.mean(diff[roi])
pscmdl /= np.mean(pscmdl[roi])
iso  /= np.mean(iso[roi])
bub  /= np.mean(bub[roi])
nfw  /= np.mean(nfw[roi])
disk /= np.mean(disk[roi])

np.save('data/fermidata/counts_Jun19_NSIDE512.npy', counts)
np.save('data/fermidata/pscmask_Jun19_NSIDE512.npy', psmask)
#np.save('data/templates_Jun19/p6v11_NSIDE512.npy', diff)
np.save('data/templates_Jun19/iso_NSIDE512.npy', iso)
np.save('data/templates_Jun19/bub_NSIDE512.npy', bub)
np.save('data/templates_Jun19/gce_NSIDE512.npy', nfw)
np.save('data/templates_Jun19/psc_NSIDE512.npy', pscmdl)
np.save('data/templates_Jun19/dsk_NSIDE512.npy', disk)

## 3. Convert to cartesian

In [10]:
# Load standard templates and convert to Cartesian
extent = 20  # Semi-extent in degrees
n_pixels = 80  # Number of pixels
pixelsize = 2 * extent / n_pixels

for name in ['gce', 'psc', 'iso', 'dsk', 'bub', 'p6v11',
          'ccwaics', 'ccwapibrem', 'ccwfics', 'ccwfpibrem']:
    temp = to_cart(np.load(f"data/templates_Jun19/{name}_NSIDE512.npy"), n_pixels=n_pixels, pixelsize=pixelsize)
    np.save(f"data/templates_Jun19/{name}_cart.npy", temp)
    print(name, end=' ')
# print()
for name in ['exposure', 'counts', 'pscmask']:
    temp = to_cart(np.load(f"data/fermidata/{name}_Jun19_NSIDE512.npy"), n_pixels=n_pixels, pixelsize=pixelsize)
    np.save(f"data/fermidata/{name}_Jun19_cart.npy", temp)
    print(name, end=' ')

gce psc iso dsk bub p6v11 ccwaics ccwapibrem ccwfics ccwfpibrem exposure counts pscmask 

## A. Counts files in TMPL_DIR_573W?
fermi-allsky-[...]-[...]GeV-fwhm000-0512-bestpsf-nopsc.fits

In [ ]:
# get filenames and eng labels
fns = [fn for fn in os.listdir(DATA_DIR_573W) if 'fwhm000-0512-bestpsf-nopsc' in fn]
eng_str_s = [fn.split('GeV')[0].split('-')[-1] for fn in fns]
eng_eng_str_s = [(float(eng_str), eng_str) for eng_str in eng_str_s]
eng_eng_str_s.sort()
eng_eng_str_s.insert(0, (0.2, '000.2'))

In [ ]:
f = fits.open(f'{TMPL_DIR_573W}/{fns[0]}')

In [ ]:
f[1].header

In [ ]:
hp.mollview(np.log(f[0].data))

In [ ]:
hp.mollview(f[1].data)

In [ ]:
f[0].header

In [ ]:
hp.mollview(f[2].data)